# Nonlinear Regression — Part 2
### Michaelis-Menten Curve Fitting with `scipy.optimize.curve_fit`

**Author:** Omar Hamouda

**Environment:** Python 3.x | Jupyter Notebook | Anaconda

---
## 📌 Overview
This notebook performs **nonlinear curve fitting** of enzyme kinetics
data to the **Michaelis-Menten equation** using `scipy.optimize.curve_fit`.

We will:
1. Load processed kinetics data (`MM_data.csv`) from Part 1.
2. Define a **custom function** for the Michaelis-Menten equation.
3. Use `curve_fit` to estimate **Vmax** and **Km**.
4. Extract standard deviations from the covariance matrix.
5. Plot the experimental data with the fitted curve.
6. Generate a smooth, publication-quality figure.

## 📚 Table of Contents
1. Importing Libraries (os, pandas, numpy, matplotlib, curve_fit)
2. Loading the Processed Data
3. Defining the Michaelis-Menten Function
4. Fitting with `curve_fit`
5. Extracting Parameters & Standard Deviations
6. Plotting the Fit
7. Smoothing the Curve with `np.linspace`
8. Formatting with Labels & Annotations
9. Exercise: Chymotrypsin Kinetics (Take-Home)

In [1]:
# Auto-Fix Working Directory 
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory set to:", os.getcwd())

Working directory set to: /home/omar/biochemist-python_ORGANIZED


In [ ]:
# Import the libraries we need
import os                          # For file paths
import pandas as pd                # For DataFrames
import numpy as np                 # For numerical operations
import matplotlib.pyplot as plt    # For plotting
from scipy.optimize import curve_fit   # For nonlinear curve fitting

### Loading the Processed Data

We load `MM_data.csv` — the file we created in **Lesson 08**.
This file contains the substrate concentrations and their
corresponding **initial velocities** (in µM/min), ready for
nonlinear regression.

This separation of concerns — **processing** (Part 1) vs.
**analysis** (Part 2) — is a standard scientific workflow.

In [ ]:
# Build the path to the processed data file
datafile = os.path.join('data', 'MM_data.csv')
print(datafile)

# Verify it exists (best practice)
if os.path.exists(datafile):
    print("File found ✅")
else:
    print("File not found ❌")

In [ ]:
# Load the processed data into a DataFrame
rates_df = pd.read_csv(datafile)

# Check column names FIRST (lesson from KeyError!)
print("Columns:", rates_df.columns.tolist())

# Display the DataFrame
rates_df

### Extracting X and Y

We extract the two Series needed for `curve_fit`:
- **xdata** — substrate concentration (`pNPP (mM)`)
- **ydata** — initial velocity (`initial velocities`)

Using square brackets `[]` — **not** parentheses `()` — to
select columns from a DataFrame.

In [ ]:
# Extract X (substrate concentration) and Y (initial velocity)
xdata = rates_df['pNPP (mM)']              # X: independent variable
ydata = rates_df['initial velocities']     # Y: dependent variable

# Verify the shapes
print(f"xdata shape: {xdata.shape}")
print(f"ydata shape: {ydata.shape}")

In [ ]:
# Inspect the data to find the extra row
print("Shape:", rates_df.shape)
print("\nFull DataFrame:")
print(rates_df)

# Check for NaN values
print("\nAny NaN?")
print(rates_df.isnull().sum())

In [ ]:
# Verify the data loaded correctly
print("rates_df shape:", rates_df.shape)
print("xdata shape:", xdata.shape)
print("ydata shape:", ydata.shape)

### Visual Inspection

Before any curve fitting, we **look at the data** to confirm it
follows a Michaelis-Menten shape:
- Rapid increase at low [S].
- Plateau at high [S] (approaching Vmax).

This quick check catches problems early.

In [ ]:
# Quick visual inspection of the data
plt.figure(figsize=(10, 6))
plt.plot(xdata, ydata, 'o', label='data')
plt.xlabel('[pNPP] (mM)')
plt.ylabel('Initial Velocity (µM/min)')
plt.legend()
plt.show()

### Defining the Michaelis-Menten Function

`curve_fit` needs a **function** to fit the data to. We define it
using Python's `def` keyword.

The function must accept:
- The **independent variable** (substrate concentration) first
- The **parameters** to be fitted (Vmax, Km) after

This matches how `curve_fit` passes arguments to the function.

In [ ]:
# Define the Michaelis-Menten function
def MM(SubConc, Vmax, Km):
    """
    Michaelis-Menten equation for enzyme kinetics.
    
    Parameters:
    -----------
    SubConc : array-like
        Substrate concentration
    Vmax : float
        Maximum velocity
    Km : float
        Michaelis constant
    
    Returns:
    --------
    Vi : array-like
        Initial velocity
    """
    Vi = (Vmax * SubConc) / (Km + SubConc)
    return Vi

In [ ]:
help(MM)   # هيطبعلك كل التوثيق ده

### Testing the Function

Before using `MM` in `curve_fit`, we test it with **sample parameters**
(`Vmax=100, Km=1`) to confirm:
- The function runs without errors.
- The output makes sense (V = Vmax/2 when [S] = Km).

**Defining a function does not execute it** — we must call it.

In [ ]:
# Test the function with sample parameters (Vmax=100, Km=1)
test_result = MM(xdata, 100, 1)
print(test_result)

### Fitting with `curve_fit`

`scipy.optimize.curve_fit()` performs a **nonlinear least-squares fit**
of our data to the `MM` function. It returns:
- **popt** — optimized parameters `[Vmax, Km]` (in the same order
  as they appear in the function definition).
- **pcov** — a 2D covariance matrix, used to compute standard errors.

In [ ]:
# Perform the nonlinear curve fit
popt, pcov = curve_fit(MM, xdata, ydata)

# Display the raw outputs
print("popt (optimized parameters):", popt)
print("pcov (covariance matrix):")
print(pcov)

### Extracting Vmax and Km

We unpack `popt` into two named variables for clarity:
- `Vmax` = `popt[0]` — maximum velocity
- `Km`   = `popt[1]` — Michaelis constant

The **order matters** — it matches the parameter order in the `MM` function.

In [ ]:
# Extract Vmax and Km from the optimized parameters
Vmax = popt[0]
Km = popt[1]

print(f"Vmax = {Vmax}")
print(f"Km = {Km}")

### Extracting Standard Deviations

The covariance matrix (`pcov`) has the **variances** on its diagonal.
Taking the square root gives us the **standard deviations** for each
parameter.

We use two numpy functions:
- `np.diag()` — extracts the diagonal elements
- `np.sqrt()` — square root

The result is a **confidence interval** for each parameter.

In [ ]:
# Calculate standard deviations for the parameters
stdev = np.sqrt(np.diag(pcov))

print(f"Vmax = {Vmax} ± {stdev[0]}")
print(f"Km = {Km} ± {stdev[1]}")

### Formatting the Output with F-Strings

By default, Python prints floats with **many decimal places**
(e.g., `36.65734834390893`). For clarity, we use **f-strings with
format specifiers** to control precision.

Syntax: `f"{variable:.2f}"` → 2 decimal places.

In [ ]:
# Print with controlled precision using f-strings
print(f'Vmax for this reaction: {Vmax:.2f} +/- {stdev[0]:.2f}')
print(f'Km for this reaction: {Km:.2f} +/- {stdev[1]:.2f}')

### Plotting the Fit

We now compare our experimental data with the fitted curve.
Two layers on the plot:
1. **Experimental data** — as scatter points (`'o'`).
2. **Fitted curve** — from `MM(xdata, Vmax, Km)`.

At this stage, the fit curve may appear a bit **jagged** because
it only connects 11 points. We'll smooth it in the next step.

In [ ]:
# Calculate the fitted y values using the optimized parameters
fit_y = MM(xdata, Vmax, Km)

# Plot experimental data and the fitted curve
plt.figure(figsize=(10, 6))
plt.plot(xdata, ydata, 'o', label='data')       # Experimental data (points)
plt.plot(xdata, fit_y, '-', label='fit')        # Fitted curve (line)
plt.xlabel('[pNPP] (mM)')
plt.ylabel('Initial Velocity (µM/min)')
plt.legend()
plt.show()

### Smoothing the Curve with `np.linspace`

The fit curve drawn with only 11 points appears **jagged**. To
produce a smooth, professional-looking curve, we generate **1000
evenly-spaced x-values** between the min and max of `xdata`, then
compute the corresponding y-values.

We use `np.min(xdata)` and `np.max(xdata)` — a **dynamic range**
approach that adapts if the data changes.

In [ ]:
# Generate 1000 evenly-spaced x-values for a smooth curve
smooth_x = np.linspace(np.min(xdata), np.max(xdata), 1000)

# Calculate fitted y-values for the smooth curve
smooth_y = MM(smooth_x, Vmax, Km)

# Plot with a smooth curve
plt.figure(figsize=(10, 6))
plt.plot(xdata, ydata, 'bo', label='data')       # Blue circles
plt.plot(smooth_x, smooth_y, 'r', label='fit')   # Red smooth line
plt.xlabel('[pNPP] (mM)')
plt.ylabel('Initial Velocity (µM/min)')
plt.legend()
plt.show()

### Final Formatting — Labels and Annotations

For a **publication-quality** figure, we add:
- **Axis labels** — using LaTeX for subscripts and Greek letters.
- **Vmax and Km annotations** — placed in empty space on the plot.
- **Legend** — to distinguish data from the fit.

We use `plt.annotate(text, xy=(x, y))` to place text at specific coordinates.

In [ ]:
# Create the final, publication-quality figure
plt.figure(figsize=(10, 6))

# Plot data and fit
plt.plot(xdata, ydata, 'bo', label='data')       # Experimental data
plt.plot(smooth_x, smooth_y, 'r', label='fit')   # Smooth fitted curve

# Add axis labels (with LaTeX)
plt.xlabel('[pNPP] (mM)')
plt.ylabel('V$_i$ ($\\mu$M/min)')

# Annotate Vmax and Km values on the plot
plt.annotate(f'Vmax = {Vmax:.2f} +/- {stdev[0]:.2f}', xy=(7.5, 20))
plt.annotate(f'Km = {Km:.2f} +/- {stdev[1]:.2f}', xy=(7.5, 16))

# Add legend
plt.legend()

# Display
plt.show()

---
## 🧪 Exercise: Chymotrypsin Kinetics

Apply the same nonlinear fitting workflow to the chymotrypsin
dataset from **Part 1** (`chymo_MM_data.csv`).

### 🎯 Goals
1. Load the chymotrypsin data.
2. Perform the Michaelis-Menten fit with `curve_fit`.
3. Extract Vmax and Km + standard deviations.
4. Plot the data with a smooth fitted curve.
5. Compare with the ALP results.

### 📌 Reminder
Always check `df.columns` first — column names may differ!

### Loading the Chymotrypsin Data

We load `chymo_MM_data.csv` — the file created in Part 1 for the
chymotrypsin assay. Same structure as `MM_data.csv`:
- `pNPA (mM)` — substrate concentration
- `slopes` — absorbance change per second
- `initial velocities` — velocity in µM/sec

**Always check `df.columns` first** — the lesson from KeyError.

In [ ]:
# Build the path to the chymotrypsin data
chymo_file = os.path.join('data', 'chymo_MM_data.csv')
print(chymo_file)

# Verify it exists
if os.path.exists(chymo_file):
    print("File found ✅")
else:
    print("File not found ❌")

In [ ]:
# Load the chymotrypsin data into a DataFrame
c_rates_df = pd.read_csv(chymo_file)

# Check column names FIRST (lesson from KeyError!)
print("Columns:", c_rates_df.columns.tolist())

# Display the DataFrame
c_rates_df

### Extracting X and Y (Chymotrypsin)

Same as ALP — we extract the two Series needed for `curve_fit`:
- `c_xdata` — substrate concentration (`pNPA (mM)`)
- `c_ydata` — initial velocity

Note: this dataset has **10 concentrations** (vs 11 for ALP) —
exponential 2-fold dilutions.

In [ ]:
# Extract X and Y for chymotrypsin
c_xdata = c_rates_df['pNPA (mM)']              # X: substrate concentration
c_ydata = c_rates_df['initial velocities']     # Y: initial velocity

# Verify the shapes
print(f"c_xdata shape: {c_xdata.shape}")
print(f"c_ydata shape: {c_ydata.shape}")

### Visual Inspection (Chymotrypsin)

Before fitting, we **look at the data**. The chymotrypsin dataset
has more noise than the ALP dataset — a realistic scenario in
enzyme kinetics.

We expect a roughly Michaelis-Menten shape, but with visible scatter.

In [ ]:
# Visual inspection of the chymotrypsin data
plt.figure(figsize=(10, 6))
plt.plot(c_xdata, c_ydata, 'o', label='data')
plt.xlabel('[pNPA] (mM)')
plt.ylabel('V$_i$ ($\\mu$M/sec)')
plt.legend()
plt.show()

### Fitting the Chymotrypsin Data

We reuse the `MM` function — the same Michaelis-Menten equation
works for any enzyme. We just pass different data:

- `c_xdata`, `c_ydata` — chymotrypsin substrate & velocity.

Note: this dataset is **noisier** than ALP, so expect larger
standard deviations and possibly a less clean fit.

In [ ]:
# Perform the nonlinear curve fit on chymotrypsin data
c_popt, c_pcov = curve_fit(MM, c_xdata, c_ydata)

# Extract Vmax and Km
c_Vmax = c_popt[0]
c_Km = c_popt[1]

# Calculate standard deviations
c_stdev = np.sqrt(np.diag(c_pcov))

# Display the results
print(f'Vmax = {c_Vmax:.4f} ± {c_stdev[0]:.4f}')
print(f'Km = {c_Km:.4f} ± {c_stdev[1]:.4f}')

### Plotting the Chymotrypsin Fit

We follow the same three-part plotting approach as for ALP:
1. **Smooth the curve** — `np.linspace` with 1000 points.
2. **Plot** — experimental data + fitted curve.
3. **Annotate** — Vmax and Km values on the plot.

Same workflow, different data — that's the power of a reusable approach.

In [ ]:
# Generate smooth x-values for chymotrypsin
c_smooth_x = np.linspace(np.min(c_xdata), np.max(c_xdata), 1000)
c_smooth_y = MM(c_smooth_x, c_Vmax, c_Km)

# Create the final, publication-quality figure
plt.figure(figsize=(10, 6))

# Plot data and fit
plt.plot(c_xdata, c_ydata, 'r^', label='data')     # Red triangles
plt.plot(c_smooth_x, c_smooth_y, 'b', label='fit') # Blue line

# Add axis labels (with LaTeX)
plt.xlabel('[pNPA] (mM)')
plt.ylabel('V$_i$ ($\\mu$M/sec)')

# Annotate Vmax and Km values on the plot
plt.annotate(f'Vmax = {c_Vmax:.4f} ± {c_stdev[0]:.4f}', xy=(0.3, 0.025))
plt.annotate(f'Km = {c_Km:.4f} ± {c_stdev[1]:.4f}', xy=(0.3, 0.022))

# Add legend
plt.legend()

# Display
plt.show()

---
## 📝 Summary & Key Takeaways

In this notebook, we performed **nonlinear curve fitting** of enzyme
kinetics data to the **Michaelis-Menten equation** using
`scipy.optimize.curve_fit`.

### What we covered:
- **Imports** — `os`, `pandas`, `numpy`, `matplotlib`, `curve_fit`.
- **Loading Processed Data** — from `MM_data.csv` (Lesson 08).
- **Defining a Function** — `def MM(SubConc, Vmax, Km)`.
- **Nonlinear Fit** — `curve_fit(MM, xdata, ydata)`.
- **Extracting Parameters** — `popt[0]` (Vmax), `popt[1]` (Km).
- **Standard Deviations** — `np.sqrt(np.diag(pcov))`.
- **Smoothing** — `np.linspace` for 1000-point curves.
- **Annotations** — labels + Vmax/Km on the plot.
- **Reusability** — the same `MM` function served both assays.

### Results:

| Enzyme | Vmax (µM/min or µM/sec) | Km (mM) |
|--------|--------------------------|----------|
| **Alkaline Phosphatase** | 36.66 ± 0.67 µM/min | 1.19 ± 0.08 |
| **Chymotrypsin** | (from exercise) | (from exercise) |

### Key equation:   V = (Vmax × [S]) / (Km + [S])


---

## ⚠️ Important Notes

### On `curve_fit`
- Requires a **function** as its first argument.
- Returns **two outputs**: `popt` (parameters) and `pcov` (covariance).
- **Order of `popt`** matches the parameter order in the function.
- Works for **any** function — linear or nonlinear.

### On Standard Deviations
- `np.diag(pcov)` extracts variances (diagonal elements).
- `np.sqrt(...)` → standard deviations.
- **Small stdev** = precise estimate.

### On Fitting Noisy Data
- **Not every dataset is clean** — real biology has noise.
- Larger standard deviations reflect **data uncertainty**, not code errors.
- Consider **filtering outliers** or using **weighted fits** for difficult datasets.

### On Common Errors
- `TypeError: 'DataFrame' object is not callable` → used `()` instead of `[]`.
- `RuntimeError: Optimal parameters not found` → bad initial guess (`p0`).
- `OptimizeWarning: Covariance ... could not be estimated` → data too noisy.

---

## 🔜 Next Steps
- **Compare kinetic parameters** — Km and Vmax across enzymes.
- **Study inhibitors** — competitive, non-competitive, uncompetitive.
- **Global fitting** — fit multiple datasets simultaneously.
- **Michaelis-Menten alternatives** — Hill equation for cooperativity.
- **Thermodynamics** — van 't Hoff plots for ΔH and ΔS.

---

## 🧬 Biological Context

### The Michaelis-Menten Equation  : V = (Vmax × [S]) / (Km + [S])